## Setup

In [1]:
import os
import optuna
from dotenv import load_dotenv
from sklearn.metrics import f1_score, recall_score
import numpy as np
import pandas as pd

from src.py_src import util
from src.py_src.models import GreatFilterModel

import warnings
pd.options.mode.copy_on_write = False
warnings.filterwarnings("ignore")

In [2]:
load_dotenv()

slided_df_path = os.path.join(os.getenv("XRAY_SLIDED_PATH"), "xray_slided.parquet")
target_class = 'target_class_in_24h'
target_flux = 'target_flux_in_24h'
target_columns = [target_class, target_flux]

buffer_limits = (5.0e-7, 4.0e-6)

df_model_input = util.create_df_model_input_opt(slided_df_path, target_columns, "xl_")

Carregando 68 colunas do arquivo Parquet...


In [3]:
df_model_input

,xl_mean_1h,xl_std_1h,xl_max_1h,xl_integ_1h,xl_log_mean_1h,xl_mean_6h,xl_std_6h,xl_max_6h,xl_integ_6h,xl_log_mean_6h,...,count_M_7D,count_X_7D,sum_class_score_7D,Bdec,Cdec,Mdec,Xdec,Edec,target_class_in_24h,target_flux_in_24h
ds,,,,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00+00:00,5.575925e-08,3.107164e-09,6.297700e-08,6.691110e-07,-7.246541,5.575925e-08,3.107164e-09,6.297700e-08,6.691110e-07,-7.246541,...,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,2,2.300000e-07
2010-01-01 00:12:00+00:00,5.574467e-08,3.800849e-09,6.297700e-08,1.337872e-06,-7.247046,5.574467e-08,3.800849e-09,6.297700e-08,1.337872e-06,-7.247046,...,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,2,2.300000e-07
2010-01-01 00:24:00+00:00,5.486561e-08,3.712223e-09,6.297700e-08,1.975162e-06,-7.253807,5.486561e-08,3.712223e-09,6.297700e-08,1.975162e-06,-7.253807,...,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,2,2.300000e-07
2010-01-01 00:36:00+00:00,5.260096e-08,5.511622e-09,6.297700e-08,2.524846e-06,-7.273224,5.260096e-08,5.511622e-09,6.297700e-08,2.524846e-06,-7.273224,...,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,2,2.300000e-07
2010-01-01 00:48:00+00:00,5.184497e-08,5.561315e-09,6.297700e-08,3.110698e-06,-7.279497,5.184497e-08,5.561315e-09,6.297700e-08,3.110698e-06,-7.279497,...,0.0,0.0,0.0,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,2,2.300000e-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-28 23:00:00+00:00,7.422856e-06,1.803084e-06,1.233765e-05,4.453713e-04,-5.140458,5.760876e-06,1.695921e-06,1.233765e-05,2.073916e-03,-5.256751,...,18.0,0.0,30524.0,3.743968e-228,2.692418,2.013134,1.304704e-18,0.000059,5,1.100000e-04
2024-12-28 23:12:00+00:00,6.574647e-06,8.230624e-07,8.542100e-06,3.944788e-04,-5.185182,5.839239e-06,1.658490e-06,1.233765e-05,2.102126e-03,-5.249811,...,18.0,0.0,30524.0,3.682086e-228,2.647916,1.979860,1.283139e-18,0.000058,5,1.100000e-04
2024-12-28 23:24:00+00:00,6.208377e-06,3.507635e-07,7.220811e-06,3.725026e-04,-5.207623,5.888654e-06,1.645739e-06,1.233765e-05,2.119915e-03,-5.245839,...,18.0,0.0,30524.0,3.621226e-228,2.604150,1.947136,1.261931e-18,0.000057,5,1.100000e-04


## Preparing Data

In [4]:
great_filter_pool = df_model_input[df_model_input[target_class] > 0].copy()

train_pct = 0.7
val_pct = (1-train_pct)/2

data = util.prepare_data(
    df_model_input=great_filter_pool,
    target_class_col=target_class,
    lambda_function=lambda lb: 1 if lb > 2 else 0,
    train_pct=train_pct,
    val_pct=val_pct,
    target_flux_col=target_flux
)

In [5]:
ratio = (np.sum(data['y']['train'] == 0)) / (np.sum(data['y']['train'] == 1))
print(f"Proporção de Classes (Neg/Pos): {ratio:.2f}")

Proporção de Classes (Neg/Pos): 0.66


## Discovery Model

In [6]:
discovery_model = GreatFilterModel(
    params={
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 5,
        'n_jobs': -1,
        'random_state': 42
    },
    buffer_limits=buffer_limits
)

In [7]:
selected_features = discovery_model.discover_top_features(
    x=data['x']['train'],
    y=data['y']['train'],
    flux_values=data['flux']['train'],
    cumulative_threshold=0.95
)

--- Quick Scan (Discovery Mode) ---
Quick Scan concluído. 40 features selecionadas (de 65).


## Hyperparameter Tuning (Optuna)

In [8]:
def objective(trial):
    params = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'n_estimators': 1000,
        'random_state': 1502,
        'n_jobs': -1,
        'early_stopping_rounds': 50,
        'device': 'cuda',

        'scale_pos_weight': trial.suggest_float("scale_pos_weight", 1.0, 5.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
        'gamma': trial.suggest_float('gamma', 0.1, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'max_delta_step': trial.suggest_int('max_delta_step', 0, 10)
    }

    model = GreatFilterModel(params=params, buffer_limits=buffer_limits, features_to_keep=selected_features)

    model.fit(
        x=data['x']['train'],
        y=data['y']['train'],
        flux_values=data['flux']['train'],
        eval_set=[(data['x']['val'], data['y']['val'])],
        verbose=False
    )

    y_pred_proba = model.predict_proba(data['x']['val'])[:, 1]
    y_pred_class = (y_pred_proba >= 0.5).astype(int)

    recall_cmx = recall_score(data['y']['val'], y_pred_class, pos_label=1)
    recall_ab = recall_score(data['y']['val'], y_pred_class, pos_label=0)
    w_ab = 1.0
    w_cmx = 5.0
    score = (w_ab * recall_ab) + (w_cmx * recall_cmx)

    return score

In [9]:
study = optuna.create_study(direction='maximize')
print("\nIniciando tuning...")
study.optimize(objective, n_trials=50)

print(f"\nBest Score: {study.best_value:.4f}")
best_params = study.best_params

best_params.update({
    'n_estimators': 1000, 'objective': 'binary:logistic',
    'eval_metric': 'logloss', 'random_state': 1502,
    'n_jobs': -1, 'early_stopping_rounds': 50
})

[I 2026-03-16 11:01:04,488] A new study created in memory with name: no-name-834b0045-bd91-4dcf-9f78-0c99eeb717e4



Iniciando tuning...


[I 2026-03-16 11:01:06,113] Trial 0 finished with value: 5.237302889368278 and parameters: {'scale_pos_weight': 3.2276929953959166, 'max_depth': 6, 'learning_rate': 0.04954511845527654, 'subsample': 0.666851376746304, 'colsample_bytree': 0.6285533694499584, 'gamma': 0.5439775481862301, 'min_child_weight': 4, 'max_delta_step': 10}. Best is trial 0 with value: 5.237302889368278.
[I 2026-03-16 11:01:08,128] Trial 1 finished with value: 5.239315876945418 and parameters: {'scale_pos_weight': 2.1493525873415296, 'max_depth': 8, 'learning_rate': 0.04079171822951051, 'subsample': 0.7935607889029836, 'colsample_bytree': 0.6348828438115302, 'gamma': 4.30451316796558, 'min_child_weight': 2, 'max_delta_step': 9}. Best is trial 1 with value: 5.239315876945418.
[I 2026-03-16 11:01:09,112] Trial 2 finished with value: 5.175792840246771 and parameters: {'scale_pos_weight': 4.1235266747568655, 'max_depth': 3, 'learning_rate': 0.0459237428488275, 'subsample': 0.8368757922743661, 'colsample_bytree': 0.66


Best Score: 5.2537


In [10]:
final_model = GreatFilterModel(params=study.best_params, buffer_limits=buffer_limits, features_to_keep=selected_features)
final_model.fit(
    x=data['x']['train'], y=data['y']['train'],
    flux_values=data['flux']['train']
)

,params,"{'colsample_bytree': 0.625489670625536, 'gamma': 2.5342276284320553, 'learning_rate': 0.02609251737287573, 'max_delta_step': 10, ...}"
,buffer_limits,None
,buffer_weight,0.2
,threshold,0.5
,features_to_keep,"['xl_log_mean_6h', 'xl_log_mean_1h', ...]"


## Threshold Tuning

In [11]:
fig = final_model.get_threshold_graph(data['x']['test'], data['y']['test'])
# display(fig)

In [12]:
final_model.optimize_threshold(data['x']['test'], data['y']['test'], target_recall=0.95)

Threshold ajustado para Recall ~0.95: 0.9575


np.float32(0.9575129)

## Results

In [ ]:
print(final_model.get_classification_report(
    data['x']['test'], data['y']['test'], target_names=['AB', 'CMX']
))

In [ ]:
fig, summary = final_model.analyze_flux_errors(
    data['x']['test'], data['y']['test'],
    flux_values=data['flux']['test'],
    buffer_limits=buffer_limits
)
display(summary)

In [ ]:
error_report = final_model.analyze_error_distribution(
    x=data['x']['test'],
    y_true=data['y']['test'],
    flux_values=data['flux']['test']
)
display(error_report)

## Features Importance

In [ ]:
features_importance = final_model.get_feature_importance()
features_importance

## Export

In [ ]:
great_filter_dir = os.getenv('GREAT_FILTER_MODELS_PATH')
save_path = os.path.join(great_filter_dir, '24h/great_filter_24h_v1.joblib')
final_model.save(save_path)